In [ ]:
# adding taxonomy information
from google.colab import drive
import pandas as pd
import requests
from tqdm import tqdm
import time

# ================= MOUNT GOOGLE DRIVE =================
drive.mount('/content/drive')

# ================= SETTINGS =================
# Update these paths to match exactly where the file is stored in your Drive
input_file = "/content/drive/MyDrive/final_pep_frames_tranls.csv"
output_file = "/content/drive/MyDrive/pep_taxonomy.csv"

BATCH_SIZE = 100
BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

# ================= LOAD =================
df = pd.read_csv(input_file, dtype=str).fillna("")

# ================= UNIQUE IDS =================
unique_ids = list(set(df["uniprot_id"].str.strip()) - {""})
print(f"Unique UniProt IDs: {len(unique_ids):,}")

# ================= FETCH FUNCTION =================
def fetch_batch(batch):
    query = " OR ".join([f"accession:{x}" for x in batch])
    params = {
        "query": query,
        "format": "json",
        "fields": "accession,organism_name,lineage",
        "size": 500
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=20)

        if r.status_code != 200:
            print(f"HTTP error: {r.status_code}")
            return {}

        data = r.json()
        result_map = {}

        for entry in data.get("results", []):
            uid = entry.get("primaryAccession")
            organism = entry.get("organism", {})
            species = organism.get("scientificName", None)
            lineage = organism.get("lineage", [])

            domain = lineage[0] if len(lineage) > 0 else None
            phylum = lineage[1] if len(lineage) > 1 else None
            class_rank = lineage[2] if len(lineage) > 2 else None
            order = lineage[3] if len(lineage) > 3 else None

            result_map[uid] = (species, domain, phylum, class_rank, order)

        return result_map

    except Exception as e:
        print("Exception:", e)
        return {}

# ================= MAIN LOOP =================
results_map = {}

for i in tqdm(range(0, len(unique_ids), BATCH_SIZE), desc="Batch fetching"):
    batch = unique_ids[i:i+BATCH_SIZE]
    res = fetch_batch(batch)
    results_map.update(res)

    # ================= LIVE PREVIEW =================
    if res:
        print("\n Batch sample:")
        for k in list(res.keys())[:3]:
            print(f"{k} → {res[k]}")
    else:
        print("\n Empty batch returned")

    time.sleep(0.1)

# ================= MAP BACK =================
def assign(uid):
    return results_map.get(uid, (None, None, None, None, None))

df[["Species", "Domain", "Phylum", "Class", "Order"]] = df["uniprot_id"].apply(
    lambda x: pd.Series(assign(x))
)

# ================= DEBUG SUMMARY =================
mapped = df["Species"].notna().sum()
total = len(df)

print(f"\n✅ Mapping success: {mapped}/{total} ({(mapped/total)*100:.2f}%)")

# ================= PREVIEW =================
print("\n🔍 First 10 rows:")
print(df[["uniprot_id", "Species", "Domain", "Phylum", "Class", "Order"]].head(10))

# ================= SAVE =================
df.to_csv(output_file, index=False)

print("\n✅ Done!")
print("Saved as:", output_file)

Mounted at /content/drive
Unique UniProt IDs: 380,950


Batch fetching:   0%|          | 1/3810 [00:01<1:34:51,  1.49s/it]


 Batch sample:
O96005 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
P07585 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
Q66K89 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')


Batch fetching:   0%|          | 2/3810 [00:02<1:34:49,  1.49s/it]


 Batch sample:
P0DPD8 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
Q6R8F2 → ('Bos taurus', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
Q8VYI0 → ('Arabidopsis thaliana', 'Eukaryota', 'Viridiplantae', 'Streptophyta', 'Embryophyta')


...

Batch fetching: 100%|█████████▉| 3809/3810 [1:37:12<00:01,  1.50s/it]


 Batch sample:
Q9NRP7 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
O00481 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
Q9BT40 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')


Batch fetching: 100%|██████████| 3810/3810 [1:37:14<00:00,  1.53s/it]


 Batch sample:
Q15842 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
Q9NXJ5 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')
Q8NGI8 → ('Homo sapiens', 'Eukaryota', 'Metazoa', 'Chordata', 'Craniata')



✅ Mapping success: 1282659/1420321 (90.31%)

🔍 First 10 rows:
  uniprot_id                          Species   Domain        Phylum  \
0     Q197F8  Invertebrate iridescent virus 3  Viruses  Varidnaviria   
1     Q6GZX2    Frog virus 3 (isolate Goorha)  Viruses  Varidnaviria   
2     Q6GZX2    Frog virus 3 (isolate Goorha)  Viruses  Varidnaviria   
3     Q6GZX2    Frog virus 3 (isolate Goorha)  Viruses  Varidnaviria   
4     Q6GZX2    Frog virus 3 (isolate Goorha)  Viruses  Varidnaviria   
5     Q6GZX0    Frog virus 3 (isolate Goorha)  Viruses  Varidnaviria   
6     Q6GZX0    Frog virus 3 (isolate Goorha)  Viruses  Varidnaviria   
7     Q91G88  Invertebrate iridescent virus 6  Viruses  Varidnaviria   
8     Q6GZW8    Frog virus 3 (isolate Goorha)  Viruses  Varidnaviria   
9     Q197F3  Invertebrate iridescent virus 3  Viruses  Varidnaviria   

          Class               Order  
0  Bamfordvirae  Nucleocytoviricota  
1  Bamfordvirae  Nucleocytoviricota  
2  Bamfordvirae  Nucleocytovir

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import pandas as pd
from Bio import Entrez
from tqdm import tqdm
import time
import re

# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================
drive.mount('/content/drive')

# =========================================================
# CONFIGURATION
# =========================================================
Entrez.email = "arin.heatup@gmail.com"

INPUT_FILE = "/content/drive/MyDrive/pep_taxonomy.csv"
OUTPUT_COLORS = "/content/itol_pep_tree_colors.txt"
OUTPUT_SYMBOLS = "/content/itol_pep_symbols.txt"

# Batch size for NCBI queries (Keep <= 40 to avoid URL length limits)
BATCH_SIZE = 40

DOMAIN_COLORS = {
    "Bacteria": "#636EFA",
    "Eukarya": "#EF553B",
    "Archaea": "#00CC96",
    "Viruses": "#AB63FA"
}

def clean_newick_name(name):
    name = str(name).strip().replace(" ", "_")
    return re.sub(r'[(),:;]', '', name)

# =========================================================
# NCBI FETCHING FUNCTIONS (WITH SLEEP TIMERS)
# =========================================================
def batch_fetch_ncbi(taxa_chunk):
    """Fetches up to 40 taxa at once using an OR-joined query to massively speed up retrieval."""
    # Build query: "Taxon1"[SCIN] OR "Taxon2"[SCIN]
    query = " OR ".join([f'"{t}"[SCIN]' for t in taxa_chunk])
    results = {}

    try:
        search_handle = Entrez.esearch(db="taxonomy", term=query, retmax=len(taxa_chunk))
        search_results = Entrez.read(search_handle)
        search_handle.close()
        time.sleep(0.35) # STRICT TIMER: 3 requests per sec max

        tax_ids = search_results.get("IdList", [])
        if not tax_ids:
            return results

        fetch_handle = Entrez.efetch(db="taxonomy", id=",".join(tax_ids), retmode="xml")
        records = Entrez.read(fetch_handle)
        fetch_handle.close()
        time.sleep(0.35) # STRICT TIMER

        for record in records:
            phylum_name, domain_name = None, None
            if record["Rank"] == "phylum": phylum_name = record["ScientificName"]
            elif record["Rank"] == "superkingdom": domain_name = record["ScientificName"]

            for ancestor in record.get("LineageEx", []):
                if ancestor["Rank"] == "phylum": phylum_name = ancestor["ScientificName"]
                elif ancestor["Rank"] == "superkingdom": domain_name = ancestor["ScientificName"]

            # Save under the scientific name
            sci_name = record["ScientificName"].lower()
            results[sci_name] = (phylum_name, domain_name)

            # Also save under synonyms to help map back to our original queries
            for syn in record.get("OtherNames", {}).get("Synonym", []):
                results[syn.lower()] = (phylum_name, domain_name)

        return results
    except Exception as e:
        time.sleep(1) # Back off if there's an error
        return {}

def get_ncbi_lineage_individual(taxon_name):
    """The slow, individual fallback fetch for difficult names."""
    try:
        search_handle = Entrez.esearch(db="taxonomy", term=taxon_name, retmode="xml")
        search_results = Entrez.read(search_handle)
        search_handle.close()
        time.sleep(0.35) # STRICT TIMER

        if not search_results["IdList"]: return None, None

        tax_id = search_results["IdList"][0]
        fetch_handle = Entrez.efetch(db="taxonomy", id=tax_id, retmode="xml")
        records = Entrez.read(fetch_handle)
        fetch_handle.close()
        time.sleep(0.35) # STRICT TIMER

        record = records[0]
        phylum_name, domain_name = None, None

        if record["Rank"] == "phylum": phylum_name = record["ScientificName"]
        elif record["Rank"] == "superkingdom": domain_name = record["ScientificName"]

        for ancestor in record.get("LineageEx", []):
            if ancestor["Rank"] == "phylum": phylum_name = ancestor["ScientificName"]
            elif ancestor["Rank"] == "superkingdom": domain_name = ancestor["ScientificName"]

        return phylum_name, domain_name
    except Exception:
        return None, None

def generate_fallbacks(raw_query):
    """Aggressive fallback variations to bypass NCBI search errors."""
    queries = [raw_query]
    no_parens = re.sub(r'\(.*?\)', '', raw_query).strip()
    no_parens = re.sub(r'\s+', ' ', no_parens)
    if no_parens and no_parens not in queries: queries.append(no_parens)

    strain_split = re.split(r'\s+(type|group|genotype|subtype|subgroup|isolate|strain|var\.|f\. sp\.|serotype|pv\.|mutant)\s+', no_parens, flags=re.IGNORECASE)[0].strip()
    if strain_split and strain_split not in queries: queries.append(strain_split)

    virus_match = re.search(r'^(.+?(?:virus|phage|viroid))(?:[\s\d].*)?$', no_parens, re.IGNORECASE)
    if virus_match:
        v_name = virus_match.group(1).strip()
        if v_name not in queries: queries.append(v_name)

    words = no_parens.split()
    if len(words) >= 2:
        binomial = f"{words[0]} {words[1]}"
        if binomial not in queries: queries.append(binomial)

    if len(words) >= 1:
        genus = words[0]
        if genus not in queries: queries.append(genus)

    return queries

# =========================================================
# 1. LOAD DATA & EXTRACT UNIQUE QUERIES
# =========================================================
print(f"Reading data from {INPUT_FILE}...")
try:
    df = pd.read_csv(INPUT_FILE)
    df.columns = df.columns.str.strip()
    col_map = {col.lower(): col for col in df.columns}
except FileNotFoundError:
    print(f"❌ Error: Could not find '{INPUT_FILE}'. Check your Google Drive path.")
    exit()

rank_hierarchy = ["species", "genus", "family", "order", "class", "phylum"]
unique_targets = set()
csv_domains = {}

# Extract only the unique queries so we don't duplicate work
for index, row in df.iterrows():
    for rank in rank_hierarchy:
        if rank in col_map and pd.notna(row[col_map[rank]]):
            val = str(row[col_map[rank]]).strip()
            if val.lower() not in ["nan", "unclassified", "unknown", "not found", ""]:
                unique_targets.add(val)
                csv_domains[val] = str(row.get(col_map.get("domain", "Domain"), "Unknown")).strip()
                break

unique_targets = list(unique_targets)
print(f"Found {len(unique_targets)} unique taxa to resolve.")

# =========================================================
# 2. PHASE 1: HIGH-SPEED BATCH RESOLUTION
# =========================================================
print("\n--- PHASE 1: High-Speed Batch Fetching ---")
resolved_cache = {}

for i in tqdm(range(0, len(unique_targets), BATCH_SIZE), desc="Batching NCBI", unit="batch"):
    chunk = unique_targets[i:i+BATCH_SIZE]
    batch_results = batch_fetch_ncbi(chunk)
    resolved_cache.update(batch_results)

# =========================================================
# 3. PHASE 2: FALLBACK RESOLUTION FOR FAILED NAMES
# =========================================================
print("\n--- PHASE 2: Processing Fallbacks ---")
final_resolved_nodes = {}

for taxon in tqdm(unique_targets, desc="Finalizing Taxa", unit="taxa"):
    phylum, ncbi_domain = None, None
    csv_domain = csv_domains.get(taxon, "Unknown")

    # 1. Check if the exact name was caught in the batch
    if taxon.lower() in resolved_cache:
        phylum, ncbi_domain = resolved_cache[taxon.lower()]

    # 2. If not, run through the aggressive fallbacks
    else:
        for attempt_query in generate_fallbacks(taxon):
            # Check cache first
            if attempt_query.lower() in resolved_cache:
                phylum, ncbi_domain = resolved_cache[attempt_query.lower()]
                break

            # If not in cache, query NCBI individually (slow but safe)
            phylum, ncbi_domain = get_ncbi_lineage_individual(attempt_query)
            if phylum:
                # Save to cache so we don't fetch it again
                resolved_cache[attempt_query.lower()] = (phylum, ncbi_domain)
                break

    # 3. Store the final result
    if phylum:
        final_domain = ncbi_domain if ncbi_domain else csv_domain
        if final_domain == "Eukaryota": final_domain = "Eukarya"

        clean_phylum = clean_newick_name(phylum)
        final_resolved_nodes[clean_phylum] = final_domain

# =========================================================
# 4. GENERATE iTOL ANNOTATION FILES
# =========================================================
print("\nGenerating fully automated iTOL files...")

if not final_resolved_nodes:
    print("⚠️ WARNING: No valid phyla were found. The output files will be empty.")
else:
    with open(OUTPUT_COLORS, "w") as fc, open(OUTPUT_SYMBOLS, "w") as fs:
        fc.write("TREE_COLORS\nSEPARATOR TAB\n")

        # --- iTOL LEGEND CONFIGURATION FOR TREE_COLORS ---
        fc.write("LEGEND_TITLE\tDomains\n")
        fc.write("LEGEND_SHAPES\t1\t1\t1\t1\n")
        fc.write("LEGEND_COLORS\t#636EFA\t#EF553B\t#00CC96\t#AB63FA\n")
        fc.write("LEGEND_LABELS\tBacteria\tEukarya\tArchaea\tViruses\n")
        # -------------------------------------------------

        fc.write("DATA\n")
        fs.write("DATASET_SYMBOL\nSEPARATOR TAB\nDATASET_LABEL\tPentapeptide Symbols\nCOLOR\t#000000\nMAXIMUM_SIZE\t50\nDATA\n")

        for phylum, domain in final_resolved_nodes.items():
            color = DOMAIN_COLORS.get(domain, "#CCCCCC")
            fc.write(f"{phylum}\tbranch\t{color}\tnormal\t5\n")
            fc.write(f"{phylum}\tlabel\t{color}\tbold\t3\n")
            fs.write(f"{phylum}\t2\t25\t{color}\t1\t1\n")

    print(f"\n✅ SUCCESS! Processed {len(final_resolved_nodes)} verified unique phyla.")
    print(f"Saved colors to: {OUTPUT_COLORS}")
    print(f"Saved symbols to: {OUTPUT_SYMBOLS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading data from /content/drive/MyDrive/pep_taxonomy.csv...


/tmp/ipykernel_50938/2649010950.py:143: DtypeWarning: Columns (5,6,16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_FILE)


Found 12848 unique taxa to resolve.

--- PHASE 1: High-Speed Batch Fetching ---


Batching NCBI: 100%|██████████| 322/322 [14:10<00:00,  2.64s/batch]



--- PHASE 2: Processing Fallbacks ---


Finalizing Taxa: 100%|██████████| 12848/12848 [1:48:27<00:00,  1.97taxa/s]


Generating fully automated iTOL files...

✅ SUCCESS! Processed 122 verified unique phyla.
Saved colors to: /content/itol_pep_tree_colors.txt
Saved symbols to: /content/itol_pep_symbols.txt


In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import pandas as pd
from Bio import Entrez
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
import requests

# =========================================================
# 1. CONFIGURATION
# =========================================================
drive.mount('/content/drive')
Entrez.email = "arin.heatup@gmail.com"

# Input Files
FILE_CQWW = "/content/CQWW_tax.csv"
FILE_PEP = "/content/drive/MyDrive/pep_taxonomy.csv"

# Output Base Paths
OUT_BASE_CQWW = "/content/CQWW_TaxID_Penetration"
OUT_BASE_PEP = "/content/Pentapeptide_TaxID_Penetration"

DOMAINS = ["Bacteria", "Eukarya", "Archaea", "Viruses"]
RANKS = ["phylum", "class", "order", "family", "genus", "species"]

DOMAIN_TAX_IDS = {
    "Bacteria": "2", "Eukarya": "2759", "Archaea": "2157", "Viruses": "10239"
}

COLORS = {
    "Bacteria": {"dark": "#636EFA", "light": "#D0D4FE"},
    "Eukarya": {"dark": "#EF553B", "light": "#FBCBC3"},
    "Archaea": {"dark": "#00CC96", "light": "#B3F5E1"},
    "Viruses": {"dark": "#AB63FA", "light": "#E6D0FE"}
}

# =========================================================
# 2. GLOBAL BASELINES
# =========================================================
def get_global_ncbi_counts():
    print("\n--- Fetching Global NCBI Baseline Counts ---")
    global_counts = {d: {r: 0 for r in RANKS} for d in DOMAINS}

    for domain, tax_id in DOMAIN_TAX_IDS.items():
        for rank in RANKS:
            query = f"txid{tax_id}[Subtree] AND {rank}[Rank]"
            try:
                search_handle = Entrez.esearch(db="taxonomy", term=query, retmax=1)
                results = Entrez.read(search_handle)
                search_handle.close()
                global_counts[domain][rank] = int(results.get("Count", 0))
                time.sleep(0.35)
            except Exception as e:
                print(f"Error fetching {domain} {rank}: {e}")

    df_global = pd.DataFrame(global_counts)
    print("\nGlobal NCBI Taxa Counts (Denominator Baseline):")
    print(df_global)
    return global_counts

# =========================================================
# 3. UNIPROT TO NCBI BRIDGING ENGINE (Bulletproofed)
# =========================================================
def batch_fetch_taxids_from_uniprot(uniprot_chunk):
    url = "https://rest.uniprot.org/uniprotkb/accessions"
    params = {
        "accessions": ",".join(uniprot_chunk),
        "fields": "organism_id"
    }
    mapping = {}
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        for entry in data.get("results", []):
            acc = str(entry.get("primaryAccession", "")).strip()
            tax_id = str(entry.get("organism", {}).get("taxonId", "")).strip()
            if acc and tax_id:
                mapping[acc] = tax_id
        return mapping
    except Exception:
        time.sleep(1)
        return {}

def batch_fetch_ncbi_by_taxid(taxid_chunk):
    results = {}

    def parse_record(record):
        lineage_dict = {r: None for r in RANKS}
        lineage_dict["domain"] = None

        curr_rank = str(record.get("Rank", "")).strip()
        if curr_rank in RANKS: lineage_dict[curr_rank] = str(record.get("ScientificName", "")).strip()
        elif curr_rank == "superkingdom": lineage_dict["domain"] = str(record.get("ScientificName", "")).strip()

        for ancestor in record.get("LineageEx", []):
            a_rank = str(ancestor.get("Rank", "")).strip()
            if a_rank in RANKS: lineage_dict[a_rank] = str(ancestor.get("ScientificName", "")).strip()
            elif a_rank == "superkingdom": lineage_dict["domain"] = str(ancestor.get("ScientificName", "")).strip()
        return lineage_dict

    try:
        fh = Entrez.efetch(db="taxonomy", id=",".join(taxid_chunk), retmode="xml")
        records = Entrez.read(fh)
        fh.close()
        time.sleep(0.35)

        for rec in records:
            parsed = parse_record(rec)
            results[str(rec["TaxId"])] = parsed
            for aka in rec.get("AkaTaxIds", []):
                results[str(aka)] = parsed
    except Exception:
        time.sleep(0.5)

    missing_ids = [tid for tid in taxid_chunk if tid not in results]
    if missing_ids:
        for tid in missing_ids:
            try:
                fh = Entrez.efetch(db="taxonomy", id=tid, retmode="xml")
                rec = Entrez.read(fh)
                fh.close()
                time.sleep(0.35)
                if rec:
                    parsed = parse_record(rec[0])
                    results[tid] = parsed
            except Exception:
                time.sleep(0.35)

    return results

def process_dataset(filepath, dataset_name, resolved_taxid_cache):
    print(f"\n--- Processing {dataset_name} ({filepath}) ---")
    try:
        df = pd.read_csv(filepath, low_memory=False)
        col_map = {str(col).lower().strip(): col for col in df.columns}
    except Exception as e:
        print(f"❌ Error loading {filepath}: {e}")
        return {d: {r: 0 for r in RANKS} for d in DOMAINS}

    if "uniprot_id" not in col_map:
        print(f"❌ ERROR: Column 'uniprot_id' not found.")
        return {d: {r: 0 for r in RANKS} for d in DOMAINS}

    # 1. EXTRACT RAW IDS & CSV DOMAINS FOR BACKUP
    uniprot_col = col_map["uniprot_id"]
    unique_uniprot_ids = list(set(df[uniprot_col].dropna().astype(str).str.strip()))

    d_col = next((col_map[c] for c in ["domain", "superkingdom", "kingdom"] if c in col_map), None)
    uniprot_to_csv_domain = {}
    if d_col:
        for _, row in df.iterrows():
            up_id = str(row[uniprot_col]).strip()
            if pd.notna(row[d_col]):
                uniprot_to_csv_domain[up_id] = str(row[d_col]).strip()

    print(f"Found {len(unique_uniprot_ids)} unique UniProt IDs. Mapping to TaxIDs...")

    # 2. MAP UNIPROT TO TAXID
    uniprot_to_taxid = {}
    UP_BATCH_SIZE = 200
    for i in tqdm(range(0, len(unique_uniprot_ids), UP_BATCH_SIZE), desc="Querying UniProt API"):
        chunk = unique_uniprot_ids[i:i+UP_BATCH_SIZE]
        mapping = batch_fetch_taxids_from_uniprot(chunk)
        uniprot_to_taxid.update(mapping)

    # 3. BUILD TAXID TO CSV DOMAIN BRIDGE
    taxid_to_csv_domain = {}
    for up_id, tid in uniprot_to_taxid.items():
        if up_id in uniprot_to_csv_domain:
            taxid_to_csv_domain[tid] = uniprot_to_csv_domain[up_id]

    unique_taxids = [tid for tid in list(set(uniprot_to_taxid.values())) if tid]
    print(f"Successfully mapped to {len(unique_taxids)} unique TaxIDs. Fetching NCBI lineages...")

    # 4. FETCH NCBI LINEAGES
    NCBI_BATCH_SIZE = 100
    for i in tqdm(range(0, len(unique_taxids), NCBI_BATCH_SIZE), desc="Querying NCBI Taxonomy"):
        chunk = unique_taxids[i:i+NCBI_BATCH_SIZE]
        missing_chunk = [tid for tid in chunk if tid not in resolved_taxid_cache]

        if missing_chunk:
            batch_results = batch_fetch_ncbi_by_taxid(missing_chunk)
            resolved_taxid_cache.update(batch_results)
            for tid in missing_chunk:
                if tid not in resolved_taxid_cache:
                    resolved_taxid_cache[tid] = None

    valid_count = sum(1 for tid in unique_taxids if resolved_taxid_cache.get(tid) is not None)
    print(f"✅ Successfully extracted {valid_count} standardized lineages from NCBI.")

    # 5. SECURE TALLYING (Domain Failsafe)
    parsed_counts = {d: {r: set() for r in RANKS} for d in DOMAINS}

    for tax_id in unique_taxids:
        ncbi_lineage = resolved_taxid_cache.get(tax_id)

        if ncbi_lineage:
            # Force pure python strings and check both NCBI and CSV for the domain
            n_domain = str(ncbi_lineage.get("domain", "")).strip()
            c_domain = str(taxid_to_csv_domain.get(tax_id, "")).strip()

            # Use NCBI domain if it exists, otherwise rescue it with the CSV domain
            domain = n_domain if n_domain.lower() not in ["", "none", "nan", "unclassified", "unknown"] else c_domain

            if domain == "Eukaryota": domain = "Eukarya"

            if domain in DOMAINS:
                for rank in RANKS:
                    val = ncbi_lineage.get(rank)
                    if val and str(val).lower() not in ["", "nan", "none", "unknown", "unclassified"]:
                        parsed_counts[domain][rank].add(str(val).strip())

    final_counts = {d: {r: len(parsed_counts[d][r]) for r in RANKS} for d in DOMAINS}

    print(f"\nSuccessfully tallied {dataset_name}:")
    for d in DOMAINS:
        print(f"  {d}: Found {final_counts[d]['phylum']} Phyla -> {final_counts[d]['species']} Species")

    return final_counts

# =========================================================
# 4. PLOTTING ENGINE
# =========================================================
def plot_taxonomic_penetration(dataset_counts, global_counts, title, base_output_path):
    fig, axes = plt.subplots(1, 6, figsize=(24, 6), sharey=True)
    fig.patch.set_facecolor('white')

    fig.suptitle(f"{title}\nNCBI Taxonomic Penetration (100% Normalized)", fontsize=22, fontweight='bold', y=1.08)

    y_pos = [3, 2, 1, 0]

    for i, rank in enumerate(RANKS):
        ax = axes[i]
        for j, domain in enumerate(DOMAINS):
            glob_val = max(global_counts[domain][rank], dataset_counts[domain][rank])
            data_val = dataset_counts[domain][rank]

            pct = (data_val / glob_val * 100) if glob_val > 0 else 0

            ax.barh(y_pos[j], 100, color=COLORS[domain]["light"], edgecolor='white', height=0.7)
            ax.barh(y_pos[j], pct, color=COLORS[domain]["dark"], edgecolor='white', height=0.7)

            label_text = f"{data_val:,} / {glob_val:,} ({pct:.2f}%)"

            if pct >= 60:
                text_x, ha, text_color = 2, 'left', 'white'
            else:
                text_x, ha, text_color = pct + 2, 'left', 'black'

            ax.text(text_x, y_pos[j], label_text, va='center', ha=ha, color=text_color, fontweight='bold', fontsize=13)

        ax.set_yticks(y_pos)
        if i == 0:
            ax.set_yticklabels(DOMAINS, fontsize=14, fontweight='bold')
        else:
            ax.tick_params(labelleft=False)

        ax.set_title(rank.capitalize(), fontsize=16, fontweight='bold')
        ax.set_xlim(0, 100)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.supxlabel("Percentage of Total NCBI Taxonomic Lineages Occupied (%)", fontsize=18, fontweight='bold', y=-0.05)
    plt.tight_layout(w_pad=1.5)

    png_path = f"{base_output_path}.png"
    tif_path = f"{base_output_path}.tif"
    plt.savefig(png_path, dpi=300, bbox_inches='tight', format='png')
    plt.savefig(tif_path, dpi=300, bbox_inches='tight', format='tiff')
    plt.close()

# =========================================================
# 5. MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    ncbi_globals = get_global_ncbi_counts()
    shared_resolved_taxid_cache = {}

    cqww_counts = process_dataset(FILE_CQWW, "CQWW Dataset", shared_resolved_taxid_cache)
    plot_taxonomic_penetration(cqww_counts, ncbi_globals, "CQWW Peptides", OUT_BASE_CQWW)

    pep_counts = process_dataset(FILE_PEP, "Pentapeptides Dataset", shared_resolved_taxid_cache)
    plot_taxonomic_penetration(pep_counts, ncbi_globals, "Total Pentapeptides", OUT_BASE_PEP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Fetching Global NCBI Baseline Counts ---

Global NCBI Taxa Counts (Denominator Baseline):
         Bacteria  Eukarya  Archaea  Viruses
phylum        192       79       46       22
class         213      320       56       51
order         481     1481       96      106
family       1134     9911      133      426
genus        5928   122813      349     4197
species    552683  1783911    13643    94990

--- Processing CQWW Dataset (/content/CQWW_tax.csv) ---
Found 440 unique UniProt IDs. Mapping to TaxIDs...


Querying UniProt API: 100%|██████████| 3/3 [00:00<00:00, 15.49it/s]


Successfully mapped to 195 unique TaxIDs. Fetching NCBI lineages...


Querying NCBI Taxonomy: 100%|██████████| 2/2 [00:02<00:00,  1.38s/it]


✅ Successfully extracted 195 standardized lineages from NCBI.

Successfully tallied CQWW Dataset:
  Bacteria: Found 11 Phyla -> 81 Species
  Eukarya: Found 10 Phyla -> 71 Species
  Archaea: Found 2 Phyla -> 5 Species
  Viruses: Found 3 Phyla -> 8 Species

--- Processing Pentapeptides Dataset (/content/drive/MyDrive/pep_taxonomy.csv) ---
Found 380950 unique UniProt IDs. Mapping to TaxIDs...


Querying UniProt API: 100%|██████████| 1905/1905 [19:02<00:00,  1.67it/s]


Successfully mapped to 13057 unique TaxIDs. Fetching NCBI lineages...


Querying NCBI Taxonomy: 100%|██████████| 131/131 [03:20<00:00,  1.53s/it]


✅ Successfully extracted 13057 standardized lineages from NCBI.

Successfully tallied Pentapeptides Dataset:
  Bacteria: Found 40 Phyla -> 2724 Species
  Eukarya: Found 54 Phyla -> 5426 Species
  Archaea: Found 9 Phyla -> 176 Species
  Viruses: Found 19 Phyla -> 1283 Species


In [ ]:
from google.colab import drive
import pandas as pd
from Bio import Entrez
from tqdm import tqdm
import time
import requests
import re

# =========================================================
# CONFIGURATION
# =========================================================
drive.mount('/content/drive')
Entrez.email = "arin.heatup@gmail.com"

INPUT_FILE = "/content/drive/MyDrive/pep_taxonomy.csv"
OUTPUT_COLORS = "/content/itol_pep_tree_colors.txt"
OUTPUT_SYMBOLS = "/content/itol_pep_symbols.txt"

DOMAIN_COLORS = {
    "Bacteria": "#636EFA",
    "Eukarya": "#EF553B",
    "Archaea": "#00CC96",
    "Viruses": "#AB63FA"
}

def clean_newick_name(name):
    name = str(name).strip().replace(" ", "_")
    return re.sub(r'[(),:;]', '', name)

# =========================================================
# UNIPROT TO NCBI BRIDGING ENGINE
# =========================================================
def batch_fetch_taxids_from_uniprot(uniprot_chunk):
    url = "https://rest.uniprot.org/uniprotkb/accessions"
    params = {"accessions": ",".join(uniprot_chunk), "fields": "organism_id"}
    mapping = {}
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        for entry in response.json().get("results", []):
            acc = str(entry.get("primaryAccession", "")).strip()
            tax_id = str(entry.get("organism", {}).get("taxonId", "")).strip()
            if acc and tax_id:
                mapping[acc] = tax_id
        return mapping
    except Exception:
        time.sleep(1)
        return {}

def batch_fetch_ncbi_by_taxid(taxid_chunk):
    results = {}

    def parse_record(record):
        phylum_name, domain_name = None, None
        curr_rank = str(record.get("Rank", "")).strip()
        if curr_rank == "phylum": phylum_name = str(record.get("ScientificName", "")).strip()
        elif curr_rank == "superkingdom": domain_name = str(record.get("ScientificName", "")).strip()

        for ancestor in record.get("LineageEx", []):
            a_rank = str(ancestor.get("Rank", "")).strip()
            if a_rank == "phylum": phylum_name = str(ancestor.get("ScientificName", "")).strip()
            elif a_rank == "superkingdom": domain_name = str(ancestor.get("ScientificName", "")).strip()
        return phylum_name, domain_name

    try:
        fh = Entrez.efetch(db="taxonomy", id=",".join(taxid_chunk), retmode="xml")
        records = Entrez.read(fh)
        fh.close()
        time.sleep(0.35)

        for rec in records:
            parsed = parse_record(rec)
            results[str(rec["TaxId"])] = parsed
            for aka in rec.get("AkaTaxIds", []):
                results[str(aka)] = parsed
    except Exception:
        time.sleep(0.5)

    missing_ids = [tid for tid in taxid_chunk if tid not in results]
    if missing_ids:
        for tid in missing_ids:
            try:
                fh = Entrez.efetch(db="taxonomy", id=tid, retmode="xml")
                rec = Entrez.read(fh)
                fh.close()
                time.sleep(0.35)
                if rec:
                    results[tid] = parse_record(rec[0])
            except Exception:
                time.sleep(0.35)
    return results

# =========================================================
# 1. LOAD DATA & EXTRACT UNIPROT IDS
# =========================================================
print(f"Reading data from {INPUT_FILE}...")
try:
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    col_map = {str(col).lower().strip(): col for col in df.columns}
except FileNotFoundError:
    print(f"❌ Error: Could not find '{INPUT_FILE}'.")
    exit()

if "uniprot_id" not in col_map:
    print(f"❌ ERROR: Column 'uniprot_id' not found in {INPUT_FILE}.")
    exit()

uniprot_col = col_map["uniprot_id"]
unique_uniprot_ids = list(set(df[uniprot_col].dropna().astype(str).str.strip()))

d_col = next((col_map[c] for c in ["domain", "superkingdom", "kingdom"] if c in col_map), None)
uniprot_to_csv_domain = {}
if d_col:
    for _, row in df.iterrows():
        up_id = str(row[uniprot_col]).strip()
        if pd.notna(row[d_col]):
            uniprot_to_csv_domain[up_id] = str(row[d_col]).strip()

print(f"Found {len(unique_uniprot_ids)} unique UniProt IDs. Mapping to TaxIDs...")

# =========================================================
# 2. RESOLVE LINEAGES
# =========================================================
uniprot_to_taxid = {}
UP_BATCH_SIZE = 200
for i in tqdm(range(0, len(unique_uniprot_ids), UP_BATCH_SIZE), desc="Querying UniProt API"):
    chunk = unique_uniprot_ids[i:i+UP_BATCH_SIZE]
    uniprot_to_taxid.update(batch_fetch_taxids_from_uniprot(chunk))

taxid_to_csv_domain = {tid: uniprot_to_csv_domain[up_id] for up_id, tid in uniprot_to_taxid.items() if up_id in uniprot_to_csv_domain}
unique_taxids = [tid for tid in list(set(uniprot_to_taxid.values())) if tid]
print(f"Mapped to {len(unique_taxids)} TaxIDs. Fetching NCBI lineages...")

resolved_taxid_cache = {}
NCBI_BATCH_SIZE = 100
for i in tqdm(range(0, len(unique_taxids), NCBI_BATCH_SIZE), desc="Querying NCBI Taxonomy"):
    chunk = unique_taxids[i:i+NCBI_BATCH_SIZE]
    batch_results = batch_fetch_ncbi_by_taxid(chunk)
    resolved_taxid_cache.update(batch_results)
    for tid in chunk:
        if tid not in resolved_taxid_cache:
            resolved_taxid_cache[tid] = (None, None)

final_resolved_nodes = {}
for tax_id in unique_taxids:
    phylum, n_domain = resolved_taxid_cache.get(tax_id, (None, None))

    if phylum and str(phylum).lower() not in ["", "nan", "none", "unknown", "unclassified"]:
        n_domain_clean = str(n_domain).strip()
        c_domain_clean = str(taxid_to_csv_domain.get(tax_id, "")).strip()

        domain = n_domain_clean if n_domain_clean.lower() not in ["", "none", "nan", "unclassified", "unknown"] else c_domain_clean
        if domain == "Eukaryota": domain = "Eukarya"

        clean_phylum = clean_newick_name(phylum)
        final_resolved_nodes[clean_phylum] = domain

# =========================================================
# 3. GENERATE iTOL ANNOTATION FILES
# =========================================================
print("\nGenerating fully automated iTOL files...")
if not final_resolved_nodes:
    print("⚠️ WARNING: No valid phyla found.")
else:
    with open(OUTPUT_COLORS, "w") as fc, open(OUTPUT_SYMBOLS, "w") as fs:
        fc.write("TREE_COLORS\nSEPARATOR TAB\n")
        fc.write("LEGEND_TITLE\tDomains\n")
        fc.write("LEGEND_SHAPES\t1\t1\t1\t1\n")
        fc.write("LEGEND_COLORS\t#636EFA\t#EF553B\t#00CC96\t#AB63FA\n")
        fc.write("LEGEND_LABELS\tBacteria\tEukarya\tArchaea\tViruses\nDATA\n")

        fs.write("DATASET_SYMBOL\nSEPARATOR TAB\nDATASET_LABEL\tPentapeptide Symbols\nCOLOR\t#000000\nMAXIMUM_SIZE\t50\nDATA\n")

        for phylum, domain in final_resolved_nodes.items():
            color = DOMAIN_COLORS.get(domain, "#CCCCCC")
            fc.write(f"{phylum}\tbranch\t{color}\tnormal\t5\n")
            fc.write(f"{phylum}\tlabel\t{color}\tbold\t3\n")
            fs.write(f"{phylum}\t2\t25\t{color}\t1\t1\n")

    print(f"✅ SUCCESS! Processed {len(final_resolved_nodes)} verified unique phyla.")
    print(f"Saved to: {OUTPUT_COLORS} & {OUTPUT_SYMBOLS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading data from /content/drive/MyDrive/pep_taxonomy.csv...
Found 380950 unique UniProt IDs. Mapping to TaxIDs...


Querying UniProt API: 100%|██████████| 1905/1905 [19:00<00:00,  1.67it/s]


Mapped to 13057 TaxIDs. Fetching NCBI lineages...


Querying NCBI Taxonomy: 100%|██████████| 131/131 [03:16<00:00,  1.50s/it]


Generating fully automated iTOL files...
✅ SUCCESS! Processed 122 verified unique phyla.
Saved to: /content/itol_pep_tree_colors.txt & /content/itol_pep_symbols.txt
